In [92]:
# CLEANING AND MISSING DATA HANDLING
# Raw monitoring data is never clean. Metrics agents go offline — nulls appear. CSV exports from ServiceNow have inconsistent formatting. Timestamps arrive as strings. Duplicate rows appear when pipelines retry. A model trained on dirty data produces confidently wrong anomaly alerts.
# Garbage in — garbage out. Cleaning is not optional.

In [93]:
# LOAD DATASETS

import pandas as pd
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (525, 7)
df_tickets : (213, 7)
df_logs    : (315, 5)


In [94]:
# DETECTING NULLS
print("Total nulls per columns:", df_metrics.isnull().sum())
print("percentage of nulls per columns:", df_metrics.isnull() / len(df_metrics) *100)
print("Any nulls at all", df_metrics.isnull().any())
print("Rows where any column has a null", df_metrics[df_metrics.isnull().any(axis=1)])

Total nulls per columns: timestamp      0
server_id      0
cpu_pct        0
memory_pct     0
response_ms    0
disk_pct       0
status         0
dtype: int64
percentage of nulls per columns:      timestamp  server_id  cpu_pct  memory_pct  response_ms  disk_pct  status
0          0.0        0.0      0.0         0.0          0.0       0.0     0.0
1          0.0        0.0      0.0         0.0          0.0       0.0     0.0
2          0.0        0.0      0.0         0.0          0.0       0.0     0.0
3          0.0        0.0      0.0         0.0          0.0       0.0     0.0
4          0.0        0.0      0.0         0.0          0.0       0.0     0.0
..         ...        ...      ...         ...          ...       ...     ...
520        0.0        0.0      0.0         0.0          0.0       0.0     0.0
521        0.0        0.0      0.0         0.0          0.0       0.0     0.0
522        0.0        0.0      0.0         0.0          0.0       0.0     0.0
523        0.0        0.0     

In [95]:
#DROPPING NULLS
print("Drop rows where any column has null", df_metrics.dropna())
print("Drop rows where specific column has null", df_metrics.dropna(subset=["cpu_pct"]))
print("Drop rows where ALL columns are null:", df_metrics.dropna(how="all"))
threshold = len(df_metrics) * 0.5
print("Drop columns with more than 50 % nulls", threshold)
df_metrics.dropna(axis=1, thresh=threshold)
#Use dropna() carefully in time series data — dropping rows breaks the time continuity. Filling is usually better than dropping for metric data.

Drop rows where any column has null               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
520 2026-01-01 04:00:00    srv-01     22.5        73.3        411.5      63.1   
521 2026-01-01 04:00:00    srv-02     39.7        57.9        918.9      44.9   
522 2026-01-01 04:00:00    srv-03     42.9        41.0       1119.2      82.5   
523 2026-01-01 04:00:00    srv-04     88.8        84.6        264.6      88.0   
524 2026-01-01 04:00:00    srv-05     83.8        90.9        415.7      

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
0,2026-01-01 00:00:00,srv-01,49.6,94.6,891.8,68.9,ok
1,2026-01-01 00:00:00,srv-02,32.3,33.9,1046.1,69.1,ok
2,2026-01-01 00:00:00,srv-03,21.6,96.0,1007.3,43.8,ok
3,2026-01-01 00:00:00,srv-04,34.5,50.7,653.5,58.1,ok
4,2026-01-01 00:00:00,srv-05,68.3,39.5,386.0,53.8,ok
...,...,...,...,...,...,...,...
520,2026-01-01 04:00:00,srv-01,22.5,73.3,411.5,63.1,warning
521,2026-01-01 04:00:00,srv-02,39.7,57.9,918.9,44.9,ok
522,2026-01-01 04:00:00,srv-03,42.9,41.0,1119.2,82.5,ok
523,2026-01-01 04:00:00,srv-04,88.8,84.6,264.6,88.0,ok


In [96]:
# Filling nulls - fillna()
print("Fill with a fixed value", df_metrics["cpu_pct"].fillna(0))
print("Fill with column mean - good for metric data", df_metrics["cpu_pct"].fillna(df_metrics["cpu_pct"].mean()))
print("Fill with column median - better when outliers exist", df_metrics["cpu_pct"].fillna(df_metrics["cpu_pct"].median()))
print("Fill with forward fill - use previous valid value", df_metrics["cpu_pct"].fillna(method="ffill"))
print("Fill with backward fill - use next valid value", df_metrics["cpu_pct"].fillna(method="bfill"))

Fill with a fixed value 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with column mean - good for metric data 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with column median - better when outliers exist 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with forward fill - use previous valid value 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with backward fill - use next valid value 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22

C:\Users\HOME\AppData\Local\Temp\ipykernel_5452\1478460071.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  print("Fill with forward fill - use previous valid value", df_metrics["cpu_pct"].fillna(method="ffill"))
C:\Users\HOME\AppData\Local\Temp\ipykernel_5452\1478460071.py:6: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  print("Fill with backward fill - use next valid value", df_metrics["cpu_pct"].fillna(method="bfill"))


In [97]:
# interpolate() - smooth gap filling
# Better than forward fill for gradual metric changes
print("Linear interpolation - fills gap with straight line between known values", df_metrics["cpu_pct"].interpolate(method="linear"))
print("Time-based interpolation - accounts for unevan time gaps")
df_metrics.set_index("timestamp")["cpu_pct"].interpolate(method="time")

Linear interpolation - fills gap with straight line between known values 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Time-based interpolation - accounts for unevan time gaps


timestamp
2026-01-01 00:00:00    49.6
2026-01-01 00:00:00    32.3
2026-01-01 00:00:00    21.6
2026-01-01 00:00:00    34.5
2026-01-01 00:00:00    68.3
                       ... 
2026-01-01 04:00:00    22.5
2026-01-01 04:00:00    39.7
2026-01-01 04:00:00    42.9
2026-01-01 04:00:00    88.8
2026-01-01 04:00:00    83.8
Name: cpu_pct, Length: 525, dtype: float64

In [98]:
# Detecting and removing duplicates
print("Check for duplicate rows:", df_metrics.duplicated().sum())

Check for duplicate rows: 25


In [99]:
print("see which rows are duplicated", df_metrics[df_metrics.duplicated()])

see which rows are duplicated               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
500 2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
501 2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
502 2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
503 2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
504 2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
505 2026-01-01 01:00:00    srv-01     82.0        43.6        641.4      68.5   
506 2026-01-01 01:00:00    srv-02     68.0        41.6        124.8      91.7   
507 2026-01-01 01:00:00    srv-03     83.9        50.7        162.3      74.5   
508 2026-01-01 01:00:00    srv-04     29.6        63.7         89.5      89.1   
509 2026-01-01 01:00:00    srv-05     72.3        51.2        648.1      65.5   
510 2026-01-01 02:00:00    srv-01     96.6        82.7       1130.4      88.2  

In [100]:
print("Duplicates based on specific columns", df_metrics.duplicated(subset=["server_id", "timestamp"]).sum())

Duplicates based on specific columns 25


In [101]:
print("Remove duplicates - keep first occurence:", df_metrics.drop_duplicates())

Remove duplicates - keep first occurence:               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
495 2026-01-05 03:00:00    srv-01     88.7        38.9        959.1      38.1   
496 2026-01-05 03:00:00    srv-02     41.8        89.6       1135.6      39.7   
497 2026-01-05 03:00:00    srv-03     97.5        62.9       1043.1      68.3   
498 2026-01-05 03:00:00    srv-04     42.6        43.8        926.1      55.1   
499 2026-01-05 03:00:00    srv-05     58.9        69.3       1045.4

In [102]:
print("Remove duplicates on specific columns", df_metrics.drop_duplicates(subset=["server_id", "timestamp"]))

Remove duplicates on specific columns               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
495 2026-01-05 03:00:00    srv-01     88.7        38.9        959.1      38.1   
496 2026-01-05 03:00:00    srv-02     41.8        89.6       1135.6      39.7   
497 2026-01-05 03:00:00    srv-03     97.5        62.9       1043.1      68.3   
498 2026-01-05 03:00:00    srv-04     42.6        43.8        926.1      55.1   
499 2026-01-05 03:00:00    srv-05     58.9        69.3       1045.4    

In [103]:
# Fixing dytpes
print("Check all dtypes:", df_metrics.dtypes)

Check all dtypes: timestamp      datetime64[ns]
server_id            category
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object


In [104]:
# string ----> datetime
df_metrics["timestamp"] = pd.to_datetime(df_metrics["timestamp"])

In [105]:
# String ---> numeric
df_metrics["cpu_pct"] = pd.to_numeric(df_metrics["cpu_pct"], errors="coerce")
# errors="coerce" → converts bad values to NaN instead of throwing error

In [106]:
# Numeric ---> category - for low-cardinality string columns
df_metrics["status"] = df_metrics["status"].astype("category")
df_metrics["server_id"] = df_metrics["server_id"].astype("category")

In [107]:
# String ---> boolean
df_tickets["resolved"] = df_tickets["resolved"].astype(bool)

In [108]:
###################Cleaning string columns################

In [109]:
print("Inconsistent category values from servicenow exports", df_tickets["category"].unique())

Inconsistent category values from servicenow exports ['Service Down' 'CPU High' 'Disk Full' 'Network Lag' 'Memory Leak']


In [110]:
# standardize - strip whitespace and lowercase
df_tickets["category"] = df_tickets["category"].str.strip().str.lower()

In [111]:
# Replacing inconsistent values
df_tickets["category"] = df_tickets["category"].replace({
    "cpu hight": "CPU High",
    "disk full": "Disk Full",
    "network lag": "Network Lag"
})

In [112]:
#####################################   Handling outliers - Identity and flag ###########################

In [113]:
# Z-score method - flag values beyond 3 standar deviations
mean = df_metrics["response_ms"].mean()
std = df_metrics["response_ms"].std()
df_metrics["response_ms_zscore"] = (df_metrics["response_ms"] - mean) / std
df_metrics["is_outlier"] = df_metrics["response_ms_zscore"].abs() > 3
print(df_metrics["is_outlier"].sum())

0


In [114]:
# IQR method — more robust than z-score for skewed distributions
Q1  = df_metrics["response_ms"].quantile(0.25)
Q3  = df_metrics["response_ms"].quantile(0.75)
IQR = Q3 - Q1

outliers = df_metrics[
    (df_metrics["response_ms"] < Q1 - 1.5 * IQR) |
    (df_metrics["response_ms"] > Q3 + 1.5 * IQR)
]
print(f"Outliers detected : {len(outliers)}")

 # outlier response times are not errors to remove. They are incidents to investigate. Flag them, don't drop them.

Outliers detected : 0


In [115]:
# APP LOGS - CLEANING SPECIFIC PATTERNS


In [116]:
print("Null error codes - expected for INFO logs, problem for ERROR logs", df_logs["error_code"].isnull().sum())

Null error codes - expected for INFO logs, problem for ERROR logs 51


In [117]:
print("ERROR rows with missing error codes - data quality issue")
error_rows_no_code = df_logs[
    (df_logs["log_level"] == "ERROR") &
    (df_logs["error_code"].isnull())
    ]

ERROR rows with missing error codes - data quality issue


In [118]:
print(f"ERROR rows missing error code: {len(error_rows_no_code)}")

ERROR rows missing error code: 8


In [119]:
print(" Fill missing error codes for non-error levels with NONE ")
df_logs["error_code"] = df_logs["error_code"].fillna("NONE")
print(df_logs["error_code"])

 Fill missing error codes for non-error levels with NONE 
0      E004
1      E003
2      E001
3      NONE
4      E002
       ... 
310    E002
311    E004
312    E002
313    E001
314    E002
Name: error_code, Length: 315, dtype: object


In [120]:
print("Strip whitespace from message column:")
df_logs["message"] = df_logs["message"].str.strip()

Strip whitespace from message column:


In [121]:
# THE CLEANING PIPELINE - STANDARD ORDER
import pandas as pd
def clean_metrics(df):
    df = df.copy()

    # Fix dtypes
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["cpu_pct"]   = pd.to_numeric(df["cpu_pct"], errors="coerce")

    # Remove duplicates
    df = df.drop_duplicates(subset=["server_id", "timestamp"])

    # Fill nulls
    df["cpu_pct"]    = df["cpu_pct"].fillna(df["cpu_pct"].median())
    df["memory_pct"] = df["memory_pct"].ffill()    # ← fixed

    # Standardize categories
    df["status"] = df["status"].str.strip().str.lower()

    # Flag outliers — don't drop
    Q1  = df["response_ms"].quantile(0.25)
    Q3  = df["response_ms"].quantile(0.75)
    IQR = Q3 - Q1
    df["is_outlier"] = (
        (df["response_ms"] < Q1 - 1.5 * IQR) |
        (df["response_ms"] > Q3 + 1.5 * IQR)
    )

    return df
df_metrics_clean = clean_metrics(df_metrics)
print(df_metrics)
print(df_metrics_clean)

              timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
520 2026-01-01 04:00:00    srv-01     22.5        73.3        411.5      63.1   
521 2026-01-01 04:00:00    srv-02     39.7        57.9        918.9      44.9   
522 2026-01-01 04:00:00    srv-03     42.9        41.0       1119.2      82.5   
523 2026-01-01 04:00:00    srv-04     88.8        84.6        264.6      88.0   
524 2026-01-01 04:00:00    srv-05     83.8        90.9        415.7      37.2   

      status  response_ms_z

In [122]:
# lAB TASK 1 — server_metrics Introduce nulls artificially, then clean them:
# import numpy as np
# df_dirty = df_metrics.copy()
# df_dirty.loc[::10, "cpu_pct"] = np.nan      # every 10th row
# df_dirty.loc[::15, "response_ms"] = np.nan  # every 15th row
# Print null counts before cleaning
# Fill cpu_pct nulls with column median
# Fill response_ms nulls with forward fill
# Print null counts after — confirm zero nulls remain

In [123]:
# introduce nulls artificially
import numpy as np
df_dirty = df_metrics.copy()
df_dirty.loc[::10, "cpu_pct"] = np.nan      # every 10th row
df_dirty.loc[::15, "response_ms"] = np.nan  # every 15th row


In [124]:
print("null counts before clearning", df_dirty.isnull().sum())

null counts before clearning timestamp              0
server_id              0
cpu_pct               53
memory_pct             0
response_ms           35
disk_pct               0
status                 0
response_ms_zscore     0
is_outlier             0
dtype: int64


In [125]:
# Fill cpu_pct nulls with column median
df_dirty["cpu_pct"] = df_dirty["cpu_pct"].fillna(df_dirty["cpu_pct"].median())

In [126]:
# Fill response_ms nulls with forward fill
df_dirty["response_ms"] = df_dirty["response_ms"].ffill()

In [127]:
print("nulls after clearning:")
print(df_dirty.isnull().sum())

nulls after clearning:
timestamp             0
server_id             0
cpu_pct               0
memory_pct            0
response_ms           1
disk_pct              0
status                0
response_ms_zscore    0
is_outlier            0
dtype: int64


In [128]:
print(df_dirty["response_ms"].head(20))

0        NaN
1     1046.1
2     1007.3
3      653.5
4      386.0
5      641.4
6      124.8
7      162.3
8       89.5
9      648.1
10    1130.4
11     275.4
12    1003.0
13     972.5
14      56.4
15      56.4
16     430.5
17     783.2
18     924.9
19     541.7
Name: response_ms, dtype: float64


In [129]:
df_dirty["response_ms"] = df_dirty["response_ms"].ffill().bfill()

In [130]:
print(df_dirty.isnull().sum())

timestamp             0
server_id             0
cpu_pct               0
memory_pct            0
response_ms           0
disk_pct              0
status                0
response_ms_zscore    0
is_outlier            0
dtype: int64


In [131]:
# LAB Task 2 — server_metrics
# Introduce duplicate rows: df_dirty = pd.concat([df_metrics, df_metrics.iloc[:20]])
# Confirm duplicate count
# Remove duplicates on server_id + timestamp
# Confirm zero duplicates remain

In [132]:
# Introduce duplicate rows: 
df_dirty = pd.concat([df_metrics, df_metrics.iloc[:20]])

In [133]:
print("confirm duplicate count:", df_dirty.duplicated().sum())

confirm duplicate count: 45


In [134]:
# remove duplicates on server id + timestamp
df_dirty = df_dirty.drop_duplicates(subset=["server_id", "timestamp"])

In [135]:
# confirm zero duplicates remain
print("confirm zero duplicates", df_dirty.duplicated().sum())
print("shape before:" , pd.concat([df_metrics, df_metrics.iloc[:20]]).shape)
print("shape after:", df_dirty.shape)

confirm zero duplicates 0
shape before: (545, 9)
shape after: (500, 9)


In [137]:
# LAB Task 3 — incidents

# Check null counts across all columns in df_tickets
# resolved_at will have nulls — are these data quality issues or expected? Write your answer as a comment
# Fill null resolved_at with a fixed string "Unresolved" — then revert, explain why this is wrong in a comment
# Correct approach: leave resolved_at nulls as NaN, filter using resolved == False instead

In [138]:
df_tickets.columns.tolist()
# rows where any columns has a null
print(df_tickets[df_tickets.isnull().any(axis=1)])
#check null counts across columns
print(df_tickets.isnull().sum())
# yes resolved_at has null valudes, 
# resolved_at is null when a ticket hasn't been resolved yet
# This is NOT a data quality issue — it's a valid business state
# Correct way to identify unresolved tickets is via resolved == False
print("unresolved tickets", df_tickets["resolved"] == False)

    ticket_id server_id      category  priority          created_at  \
0     INC0001    srv-04  service down         2 2026-03-24 02:00:00   
2     INC0003    srv-03  service down         2 2026-04-02 10:00:00   
3     INC0004    srv-02  service down         1 2026-03-13 20:00:00   
5     INC0006    srv-03   Network Lag         1 2026-01-29 23:00:00   
8     INC0009    srv-02      cpu high         2 2026-01-09 08:00:00   
..        ...       ...           ...       ...                 ...   
203   INC0001    srv-04  service down         2 2026-03-24 02:00:00   
205   INC0003    srv-03  service down         2 2026-04-02 10:00:00   
206   INC0004    srv-02  service down         1 2026-03-13 20:00:00   
208   INC0006    srv-03   Network Lag         1 2026-01-29 23:00:00   
211   INC0009    srv-02      cpu high         2 2026-01-09 08:00:00   

    resolved_at  resolved  
0           NaT     False  
2           NaT     False  
3           NaT     False  
5           NaT     False  
8      

In [ ]:
# Task 4 — incidents

# Introduce inconsistent category values:

# pythondf_tickets_dirty = df_tickets.copy()
# df_tickets_dirty.loc[::5, "category"] = "cpu high"
# df_tickets_dirty.loc[::7, "category"] = " Disk Full "

# Print value_counts() before cleaning
# Standardize with str.strip().str.lower()
# Print value_counts() after — confirm categories are consistent

In [139]:
# Introduce inconsistent category volues:
df_tickets_dirty = df_tickets.copy()
df_tickets_dirty.loc[::5, "category"] = "cpu high"
df_tickets_dirty.loc[::7, "category"] = " Disk Full "

In [140]:
# value counts
print("value counts before cleaning", df_tickets_dirty["category"].value_counts())

value counts before cleaning category
cpu high        69
Disk Full       36
 Disk Full      31
Network Lag     29
service down    26
memory leak     22
Name: count, dtype: int64


In [141]:
# Standardize with str.strip().str.lower()
df_tickets_dirty.columns.tolist()
df_tickets_dirty.dtypes
df_tickets_dirty["category"] = df_tickets_dirty["category"].str.strip().str.lower()
# Print value_counts() after — confirm categories are consistent
print(df_tickets_dirty["category"].value_counts())

category
cpu high        69
disk full       67
network lag     29
service down    26
memory leak     22
Name: count, dtype: int64


In [ ]:
# Task 5 — app_logs

# Check null counts in df_logs
# error_code will have nulls — explain in a comment whether these are expected
# Fill null error_code with "NONE" for INFO and WARNING rows only
# Leave ERROR and CRITICAL null error codes unfilled — flag them separately as data quality issues

In [160]:
print(df_logs.dtypes)
print(df_logs.isnull().sum())
# error_code can be null for INFO and WARNING logs because these levels
# typically do not represent failures. They are informational or cautionary.
# However, for ERROR and CRITICAL logs, error_code SHOULD be present.
# Missing error_code in these cases indicates a data quality issue.
print(df_logs["error_code"].value_counts())

timestamp     datetime64[ns]
server_id             object
log_level             object
message               object
error_code            object
dtype: object
timestamp     0
server_id     0
log_level     0
message       0
error_code    0
dtype: int64
error_code
E001    78
E003    72
E002    66
NONE    51
E004    48
Name: count, dtype: int64


In [154]:
dq_issues = df_logs[
    (df_logs["error_code"].isna()) &
    (df_logs["log_level"].isin(["ERROR", "CRITICAL"]))
]
print(dq_issues)

Empty DataFrame
Columns: [timestamp, server_id, log_level, message, error_code]
Index: []


In [ ]:
# Task 6 — server_metrics

# Use IQR method to detect outliers in response_ms
# Print outlier count
# Flag them with is_outlier column — do not drop
# Print value_counts() of server_id among outlier rows — which server has most outliers?

In [ ]:
Q1 = df_metrics["response_ms"].quantile(0.25)
Q3 = df_metrics["response_ms"].quantile(0.75)
IQR = Q3 - Q1

In [157]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outlier_mask = (
    (df_metrics["response_ms"] < lower_bound) |
    (df_metrics["response_ms"] > upper_bound)
)
outlier_count = outlier_mask.sum()
print("Outlier count:", outlier_count)

Outlier count: 0


In [158]:
df_metrics["is_outlier"] = outlier_mask

In [159]:
outlier_servers = df_metrics[df_metrics["is_outlier"]]["server_id"].value_counts()
print(outlier_servers)

server_id
srv-01    0
srv-02    0
srv-03    0
srv-04    0
srv-05    0
Name: count, dtype: int64


In [171]:
# Task 7 — cross-dataset

# Write a clean_logs() function that:

# Fixes timestamp dtype
# Strips whitespace from message
# Fills null error_code with "NONE"
# Removes duplicate rows
# Returns cleaned DataFrame
# Apply it to df_logs using .copy() first
# Print shape before and after

In [172]:
print("shape before", df_logs.shape)

def clean_logs(df_logs):
    df = df_logs.copy()

    # Fix timestamp dtype
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Strip whitespace from message
    df["message"] = df["message"].str.strip()

    # Fill null error_code
    df["error_code"] = df["error_code"].fillna("NONE")

    # Remove duplicates
    df = df.drop_duplicates()

    return df

df_cleaned = clean_logs(df_logs.copy())

print("shape after", df_cleaned.shape)

shape before (315, 5)
shape after (300, 5)
